## Rights agent evaluation register

A stratified benchmark of question and answer pairs for RAGAS evaluation of the
Rights agent, following supervisory guidance. Items are stratified across
in-force provisions, provisions enacted but not yet commenced, questions
spanning more than one section, and out-of-scope queries testing abstention.

Each item records the question, its thematic category, the section that answers
it, the commencement status at the time of annotation, and a gold-standard
answer drawn from the statutory text.

Corpus snapshot: Renters' Rights Act 2025 as retrieved 19 November 2026.

In [1]:
import json
import pandas as pd

In [3]:
with open('../data/processed/act_chunks.json', encoding='utf-8') as f:
    chunks = json.load(f)

acts = pd.DataFrame(chunks)

In [4]:
print("Sections:", len(acts))
print("In force:", acts['in_force_now'].sum())
print("Uncommenced:", (~acts['in_force_now']).sum())

Sections: 149
In force: 98
Uncommenced: 51


In [5]:
pd.set_option('display.max_rows', 200)
print(acts[acts['in_force_now']][['section', 'heading']].to_string(index=False))

 section                                                                       heading
       1       Assured tenancies to be periodic with rent period not exceeding a month
       2                                      Abolition of assured shorthold tenancies
       3                                             Changes to grounds for possession
       4                        Possession for anti-social behaviour: relevant factors
       5                                  Form of notice of proceedings for possession
       6                                     Statutory procedure for increases of rent
       7                                        Challenging amount or increase of rent
       8 Prohibition of rent in advance after lease entered into (except initial rent)
       9                      Prohibition of rent in advance before lease entered into
      10                          Repayment of rent paid for days after end of tenancy
      11                                   

In [6]:
testset = []

def item(qid, theme, question, phrasing, section, commencement, gold):
    testset.append({
        'id': qid, 'theme': theme, 'question': question,
        'phrasing': phrasing, 'section': section,
        'commencement': commencement, 'gold_answer': gold,
    })


# --- Tenancy structure and possession ---
item('Q01', 'tenancy_structure',
     'Can my landlord give me a fixed-term tenancy of twelve months?',
     'plain', 1, 'in_force',
     'No. Section 1 provides that terms of an assured tenancy are of no effect '
     'so far as they provide for a fixed term, and that the rent period must '
     'not exceed a month.')

item('Q02', 'tenancy_structure',
     'What is the effect of the Act on assured shorthold tenancies?',
     'statutory', 2, 'in_force',
     'Section 2 abolishes assured shorthold tenancies.')

item('Q03', 'possession',
     'My landlord wants me out. What notice do they have to give?',
     'plain', 5, 'in_force',
     'Section 5 prescribes the form of notice of proceedings for possession. '
     'The notice period depends on the ground relied upon, which section 3 sets out.')

item('Q04', 'possession',
     'What factors must a court consider when a landlord seeks possession for '
     'anti-social behaviour?',
     'statutory', 4, 'in_force',
     'Section 4 sets out the relevant factors a court must consider when '
     'determining whether to grant possession on anti-social behaviour grounds.')

# --- Rent ---
item('Q05', 'rent_increases',
     'Can my landlord put the rent up whenever they like?',
     'plain', 6, 'in_force',
     'No. Section 6 establishes a statutory procedure for rent increases. Rent '
     'for a period may not exceed the rent for the previous period except by '
     'notice under that section, by tribunal determination under section 14, or '
     'by written agreement following such a determination.')

item('Q06', 'rent_increases',
     'How may a tenant challenge the amount of a proposed rent increase?',
     'statutory', 7, 'in_force',
     'Section 7 provides for challenging the amount or increase of rent by '
     'referral to the appropriate tribunal.')

item('Q07', 'rent_in_advance',
     'My landlord is asking for six months rent up front. Is that allowed?',
     'plain', 9, 'in_force',
     'Section 9 prohibits requiring rent in advance before a lease is entered '
     'into. Section 8 prohibits rent in advance after the lease is entered '
     'into, other than the initial rent.')

item('Q08', 'rent_in_advance',
     'What is the position on rent paid for days following the end of a tenancy?',
     'statutory', 10, 'in_force',
     'Section 10 provides for repayment of rent paid in respect of days after '
     'the tenancy has ended.')

# --- Deposits and fees ---
item('Q09', 'deposits',
     'How much deposit can a landlord ask for?',
     'plain', 26, 'in_force',
     'Section 26 amends the tenancy deposit requirements. The statutory cap of '
     'five weeks rent for annual rents under £50,000 derives from the Tenant '
     'Fees Act 2019.')

item('Q10', 'fees',
     'What fees may a landlord charge in connection with a tenancy?',
     'statutory', 27, 'in_force',
     'Section 27 amends the provisions governing tenant fees.')

# --- Pets ---
item('Q11', 'pets',
     'Am I allowed to keep a cat in my rented flat?',
     'plain', 11, 'in_force',
     'Section 11 gives a tenant the right to request permission to keep a pet. '
     'The landlord must not unreasonably refuse.')

item('Q12', 'council_tax',
     'Who is liable for council tax under an assured tenancy?',
     'statutory', 28, 'in_force',
     'Section 28 provides for the liability of tenants under assured tenancies '
     'for council tax.')

print(len(testset), "items")

12 items


In [7]:
def show(n, chars=1800):
    row = acts[acts['section'] == n].iloc[0]
    print(f"=== s.{n}: {row['heading']} ===")
    print(f"Status: {'in force' if row['in_force_now'] else 'NOT COMMENCED'}"
          f"{' (partial)' if row['partial'] else ''}")
    print()
    print(row['text'][:chars])
    print()

for n in [1, 2, 5, 4]:
    show(n)

=== s.1: Assured tenancies to be periodic with rent period not exceeding a month ===
Status: in force (partial)

1 Assured tenancies to be periodic with rent period not exceeding a month
In the 1988 Act, before section 5 insert—
“4A Assured tenancies to be periodic with rent period not exceeding a 
month
(1) Terms of an assured tenancy are of no effect so far as they provide—
(a) for a tenancy to be a fixed term tenancy, or
Part 1 – Tenancy reform
Chapter 1 – Assured tenancies
for the Renters’ Rights Act 2025. (See end of Document for details)
(b) for periods of the tenancy to be different from the periods for which 
rent is payable (“rent periods”).
(2) Where terms of an assured tenancy are of no effect by virtue of 
subsection (1)(a) or (b), the tenancy has effect as a periodic tenancy under 
which the periods of the tenancy are the same as the rent periods.
(3) Terms of an assured tenancy which provide for the rent periods are of no effect 
unless each rent period is—
(a) a period o

In [8]:
testset = []

def item(qid, theme, question, phrasing, section, commencement, gold, answerable=True):
    testset.append({
        'id': qid, 'theme': theme, 'question': question,
        'phrasing': phrasing, 'section': section,
        'commencement': commencement, 'gold_answer': gold,
        'answerable': answerable,
    })


# --- Tenancy structure ---
item('Q01', 'tenancy_structure',
     'Can my landlord give me a fixed-term tenancy of twelve months?',
     'plain', 1, 'partial',
     'No. Section 1 inserts section 4A into the Housing Act 1988, providing '
     'that terms of an assured tenancy providing for a fixed term are of no '
     'effect. The tenancy instead takes effect as a periodic tenancy, and each '
     'rent period must be 28 days or shorter, or monthly.')

item('Q02', 'tenancy_structure',
     'What does the Act do to assured shorthold tenancies?',
     'statutory', 2, 'partial',
     'Section 2 omits Chapter 2 of Part 1 of the Housing Act 1988, which '
     'governed assured shorthold tenancies, and omits section 6A on demotion.')

item('Q03', 'possession',
     'What does the Act change about the form of a possession notice?',
     'statutory', 5, 'partial',
     'Section 5 amends section 8 of the Housing Act 1988 so that regulations '
     'may provide for the possession notice form to be published by the '
     'Secretary of State, and that the version in effect at the time applies.')

item('Q04', 'possession',
     'What must a court consider where possession is sought for anti-social '
     'behaviour against a tenant in an HMO?',
     'statutory', 4, 'partial',
     'Section 4 amends section 9A of the Housing Act 1988 to add whether the '
     'person co-operated with the landlord\'s attempts to stop the conduct, and '
     'requires the court to have particular regard to the effect on other '
     'occupiers sharing accommodation or facilities in the HMO.')

In [9]:
for n in [6, 7, 8, 9, 10, 11, 26, 27, 28]:
    show(n, 1500)

=== s.6: Statutory procedure for increases of rent ===
Status: in force (partial)

6 Statutory procedure for increases of rent
(1) Section 13 of the 1988 Act (increases of rent) is amended in accordance with
subsections (2) to (8).
(2) In the heading for “periodic tenancies” substitute “tenancies other than relevant low-
cost tenancies”.
(3) For subsection (1) substitute—
“(1) This section applies to any assured tenancy other than a relevant low-cost 
tenancy.”
(4) In subsection (2)—
(a) in paragraph (a), for “the minimum period” substitute “two months”;
(b) in paragraph (b)—
(i) for the words before sub-paragraph (i) substitute “either”;
(ii) after sub-paragraph (i) insert “or”;
(c) in paragraph (c)—
(i) in the words before sub-paragraph (i), after “below” insert “, either”;
(ii) after sub-paragraph (i) insert “or”.
(5) Omit subsection (3).
(6) In subsection (4)—
(a) in paragraph (a), for “by an application in the prescribed form refers the notice 
to the appropriate tribunal” substit

In [10]:
item('Q05', 'rent_increases',
     'Can my landlord put the rent up whenever they like?',
     'plain', 6, 'partial',
     'No. Section 6 amends section 13 of the Housing Act 1988 to insert '
     'subsection (4A), under which the rent for a period may not exceed the '
     'rent for the previous period except by notice under that section, by '
     'agreement following such a notice, or by tribunal determination. The '
     'notice period is two months.')

item('Q06', 'rent_increases',
     'Within what period of the tenancy may a tenant challenge the rent itself '
     'rather than a proposed increase?',
     'statutory', 7, 'partial',
     'Section 7 inserts subsection (A1) into section 14 of the Housing Act '
     '1988, allowing a tenant to challenge the rent payable. No application may '
     'be made if more than six months have elapsed since the beginning of the '
     'tenancy, or if the rent follows a previous determination.')

item('Q07', 'rent_in_advance',
     'My landlord wants six months rent before I sign. Is that allowed?',
     'plain', 9, 'partial',
     'No. Section 9 amends the Tenant Fees Act 2019 so that a payment of rent '
     'payable before an assured tenancy is entered into is a prohibited '
     'payment, and inserts section 5A prohibiting a landlord from inviting, '
     'encouraging or accepting such a payment.')

item('Q08', 'rent_in_advance',
     'Does the prohibition on rent in advance apply to tenancies that began '
     'before the Act commenced?',
     'statutory', 8, 'partial',
     'No. Section 8 inserts section 4B into the Housing Act 1988, and '
     'subsection (2)(a) excludes tenancies entered into before the commencement '
     'date. Excepted tenancies and initial rent due during the permitted '
     'pre-tenancy period are also excluded.')

item('Q09', 'rent_repayment',
     'I moved out early. Can I get back the rent I had already paid for the '
     'rest of the month?',
     'plain', 10, 'partial',
     'Yes. Section 10 inserts section 14ZC into the Housing Act 1988, entitling '
     'a tenant to repayment of any part of the rent relating to days falling '
     'after the end of the tenancy.')

item('Q10', 'pets',
     'How long does my landlord have to answer if I ask to keep a cat?',
     'plain', 11, 'partial',
     'Section 11 inserts section 16A into the Housing Act 1988. The landlord '
     'must give or refuse consent in writing on or before the 28th day after '
     'the request, and consent must not be unreasonably refused. The period may '
     'be extended where the landlord reasonably requests further information or '
     'must seek a superior landlord\'s consent.')

item('Q11', 'deposits',
     'What does the Act change about tenancy deposit protection?',
     'statutory', 26, 'partial',
     'Section 26 amends Chapter 4 of Part 6 of the Housing Act 2004, '
     'substituting "assured tenancy" for "shorthold tenancy" throughout, so '
     'that deposit protection requirements apply to assured tenancies.')

item('Q12', 'council_tax',
     'Does the Act make tenants liable for council tax?',
     'statutory', 28, 'partial',
     'Section 28 amends the definition of "material interest" in section 6(6) '
     'of the Local Government Finance Act 1992 to include a tenancy that is or '
     'was previously an assured tenancy.')

print(len(testset), "items")

12 items


In [11]:
import sys
sys.path.append('../src/agents')
from rights_agent import retrieve

for it in testset[:12]:
    hits = retrieve(it['question'], k=3)
    found = [h['section'] for h in hits]
    mark = 'OK ' if it['section'] in found else '** '
    print(f"{mark}{it['id']} expected s.{it['section']:<3} got {found}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

OK Q01 expected s.1   got [1, 31, 7]
OK Q02 expected s.2   got [2, 26, 30]
OK Q03 expected s.5   got [5, 13, 3]
OK Q04 expected s.4   got [4, 105, 98]
** Q05 expected s.6   got [8, 11, 1]
OK Q06 expected s.7   got [7, 1, 6]
** Q07 expected s.9   got [8, 1, 56]
OK Q08 expected s.8   got [8, 9, 2]
** Q09 expected s.10  got [8, 103, 9]
OK Q10 expected s.11  got [11, 77, 86]
OK Q11 expected s.26  got [26, 28, 134]
OK Q12 expected s.28  got [28, 42, 107]


In [12]:
for n in [33, 34, 35, 38, 40, 41]:
    show(n, 1600)

=== s.33: Discrimination relating to children ===
Status: in force (partial)

33 Discrimination relating to children
(1) A relevant person must not, in relation to a dwelling that is to be let on an agreement 
which may give rise to a relevant tenancy—
(a) on the basis that a child would or may live with or visit a person at the dwelling 
if the dwelling were the person’s home, prevent the person from—
(i) enquiring whether the dwelling is available for let,
(ii) accessing information about the dwelling,
(iii) viewing the dwelling in order to consider whether to seek to rent it, or
(iv) entering into a tenancy of the dwelling, or
(b) apply a provision, criterion or practice in order to make people who would 
have a child live with or visit them at the dwelling, if it were their home, less 
likely to enter into a tenancy of the dwelling than people who would not.
(2) Subsection (1) does not apply if—
Renters’ Rights Act 2025 (c. 26)
Chapter 3 – Discrimination in the rental market: Engla

In [13]:
item('Q13', 'discrimination_children',
     'Can a landlord refuse to let me view a flat because I have children?',
     'plain', 33, 'partial',
     'No. Section 33(1)(a) prohibits a relevant person from preventing someone '
     'enquiring about, accessing information on, viewing, or entering into a '
     'tenancy of a dwelling on the basis that a child would or may live with or '
     'visit them. Exceptions apply where the conduct is a proportionate means '
     'of achieving a legitimate aim, or where an insurance term requires it.')

item('Q14', 'discrimination_children',
     'What constitutes indirect discrimination relating to children under the Act?',
     'statutory', 33, 'partial',
     'Section 33(1)(b) prohibits applying a provision, criterion or practice in '
     'order to make people who would have a child live with or visit them less '
     'likely to enter into a tenancy than people who would not.')

item('Q15', 'discrimination_benefits',
     'I claim Universal Credit. Can a letting agent turn me away for that?',
     'plain', 34, 'partial',
     'No. Section 34(1) prohibits a relevant person from preventing a benefits '
     'claimant from enquiring about, accessing information on, viewing or '
     'entering into a tenancy, or from applying a provision, criterion or '
     'practice making benefits claimants less likely to obtain a tenancy.')

item('Q16', 'discrimination_terms',
     'My tenancy agreement says no children. Is that term valid?',
     'plain', 35, 'in_force',
     'No. Section 35(1) provides that such a term is of no effect so far as it '
     'prohibits or restricts a tenant having a child live with or visit them, '
     'unless it is a proportionate means of achieving a legitimate aim or '
     'required by an insurance term to which section 38 does not apply.')

item('Q17', 'discrimination_insurance',
     'Are insurance terms requiring landlords to exclude families enforceable?',
     'statutory', 38, 'in_force',
     'No. Section 38 provides that a term of a contract of insurance requiring '
     'the insured to prohibit or restrict a tenant having a child live with or '
     'visit them, or to prohibit a tenant being a benefits claimant, is of no '
     'effect. It applies to contracts entered into or extended on or after the '
     'day the section comes into force.')

item('Q18', 'discrimination_enforcement',
     'What happens to a landlord who breaks the rules on discrimination?',
     'plain', 40, 'partial',
     'Section 40(1) allows a local housing authority to impose a financial '
     'penalty where satisfied on the balance of probabilities that a person has '
     'breached section 33 or 34. Repeated penalties for continuing conduct are '
     'possible after 28 days.')

item('Q19', 'affordability_income',
     'Can a landlord still check whether I can afford the rent?',
     'plain', 41, 'in_force',
     'Yes. Section 41 provides that nothing in Chapter 3 prohibits taking a '
     'person\'s income into account when considering whether they would be able '
     'to afford to pay rent under a relevant tenancy.')

print(len(testset), "items")

19 items


In [14]:
for it in testset[12:]:
    hits = retrieve(it['question'], k=3)
    found = [h['section'] for h in hits]
    mark = 'OK ' if it['section'] in found else '** '
    print(f"{mark}{it['id']} expected s.{it['section']:<3} got {found}")

OK Q13 expected s.33  got [33, 50, 51]
OK Q14 expected s.33  got [50, 44, 33]
** Q15 expected s.34  got [103, 99, 41]
** Q16 expected s.35  got [51, 33, 36]
OK Q17 expected s.38  got [52, 38, 35]
** Q18 expected s.40  got [67, 33, 39]
OK Q19 expected s.41  got [103, 41, 7]


In [15]:
for it in testset:
    hits = retrieve(it['question'], k=3)
    found = [h['section'] for h in hits]
    rank = found.index(it['section']) + 1 if it['section'] in found else None
    mark = f"rank {rank}" if rank else "MISS  "
    print(f"{mark}  {it['id']} expected s.{it['section']:<3} got {found}")

rank 1  Q01 expected s.1   got [1, 31, 7]
rank 1  Q02 expected s.2   got [2, 26, 30]
rank 1  Q03 expected s.5   got [5, 13, 3]
rank 1  Q04 expected s.4   got [4, 105, 98]
MISS    Q05 expected s.6   got [8, 11, 1]
rank 1  Q06 expected s.7   got [7, 1, 6]
MISS    Q07 expected s.9   got [8, 1, 56]
rank 1  Q08 expected s.8   got [8, 9, 2]
MISS    Q09 expected s.10  got [8, 103, 9]
rank 1  Q10 expected s.11  got [11, 77, 86]
rank 1  Q11 expected s.26  got [26, 28, 134]
rank 1  Q12 expected s.28  got [28, 42, 107]
rank 1  Q13 expected s.33  got [33, 50, 51]
rank 3  Q14 expected s.33  got [50, 44, 33]
MISS    Q15 expected s.34  got [103, 99, 41]
MISS    Q16 expected s.35  got [51, 33, 36]
rank 2  Q17 expected s.38  got [52, 38, 35]
MISS    Q18 expected s.40  got [67, 33, 39]
rank 2  Q19 expected s.41  got [103, 41, 7]


In [16]:
print(acts[acts['section'].between(30, 56)][['section','heading']].to_string(index=False))

 section                                                                   heading
      30                 Powers of Secretary of State in connection with Chapter 1
      31                            Long tenancies and financial services products
      32                             Accommodation for homeless people or students
      33                                       Discrimination relating to children
      34                                Discrimination relating to benefits status
      35 Discriminatory terms in a tenancy relating to children or benefits status
      36          Terms in superior leases relating to children or benefits status
      37                Terms in mortgages relating to children or benefits status
      38      Terms in insurance contracts relating to children or benefits status
      39                         Power of the Secretary of State to protect others
      40          Financial penalties for breach of anti-discrimination provisions
    

In [17]:
unc = acts[~acts['in_force_now']]
print(unc[unc['section'].between(60, 100)][['section', 'heading']].to_string(index=False))

 section                                                                        heading
      60                   Remedying of hazards occurring in dwelling-houses in England
      61            Remedying of hazards occurring in accommodation in England occupied
      64                                                       Landlord redress schemes
      65                           Approval and designation of landlord redress schemes
      66                                                            Financial penalties
      67                                                                       Offences
      68 Decision under a landlord redress scheme may be made enforceable as if it were
      69                                      Landlord redress schemes: no Crown status
      70                  Guidance for scheme administrator and local housing authority
      71                                                    Interpretation of Chapter 2
      72                        

In [18]:
for n in [64, 75, 82, 90, 83]:
    show(n, 1400)

=== s.64: Landlord redress schemes ===
Status: NOT COMMENCED (partial)

64 Landlord redress schemes
(1) The Secretary of State may make regulations requiring a residential landlord to be a 
member of a landlord redress scheme.
(2) A “landlord redress scheme” means a scheme—
(a) which provides for a complaint made by or on behalf of a prospective, 
current or former residential tenant against a member of the scheme to be 
independently investigated and determined by an independent individual, and
(b) which is—
(i) approved by the Secretary of State for the purposes of regulations 
under subsection (1), or
(ii) administered by or on behalf of the Secretary of State and designated 
by the Secretary of State for those purposes.
(3) A scheme must not be approved or designated under subsection (2)(b) unless it 
satisfies the conditions set out in regulations made under section 65(1).
(4) In subsection (2)(a) “prospective residential tenant” means a person who—
(a) offers to become a resident

In [19]:
item('Q20', 'redress', 
     'Is there an ombudsman I can complain to about my landlord?',
     'plain', 64, 'uncommenced',
     'Section 64 enables the Secretary of State to make regulations requiring '
     'residential landlords to join a landlord redress scheme, under which a '
     'complaint by a prospective, current or former tenant is independently '
     'investigated. The section is enacted but not yet commenced, so no such '
     'requirement is currently in force.')

item('Q21', 'database',
     'What will the private rented sector database contain?',
     'statutory', 75, 'uncommenced',
     'Section 75 requires the database operator to establish a database with '
     'entries for residential landlords, for dwellings let under residential '
     'tenancies, and for persons subject to banning orders, convictions or '
     'financial penalties. Entries may be active or inactive. The section is '
     'enacted but not yet commenced.')

item('Q22', 'database',
     'Can a landlord advertise a property without being registered?',
     'plain', 82, 'uncommenced',
     'Section 82(1) provides that a person must not market a dwelling unless '
     'there is an active landlord entry and an active dwelling entry in the '
     'database, and subsection (2) requires written advertisements to include '
     'the unique identifiers. The section is enacted but not yet commenced, so '
     'the restriction does not yet apply.')

item('Q23', 'database',
     'Can my landlord evict me if they are not on the database?',
     'plain', 90, 'uncommenced',
     'Section 90 inserts subsection (5ZC) into section 7 of the Housing Act '
     '1988, preventing a court making a possession order while the landlord is '
     'in breach of the database registration duty, except on Ground 7A or '
     'Ground 14. The section is enacted but not yet fully commenced.')

item('Q24', 'database',
     'When must a local housing authority enter a landlord on the database '
     'following a banning order?',
     'statutory', 83, 'uncommenced',
     'Section 83(1) requires a local housing authority to make an entry where a '
     'relevant banning order has been made following its application, where the '
     'person has been convicted following its proceedings, or where it has '
     'imposed a financial penalty for a relevant banning order offence. The '
     'section is enacted but not yet commenced.')

In [20]:
for it in testset[19:]:
    hits = retrieve(it['question'], k=3)
    found = [h['section'] for h in hits]
    rank = found.index(it['section']) + 1 if it['section'] in found else None
    mark = f"rank {rank}" if rank else "MISS  "
    print(f"{mark}  {it['id']} expected s.{it['section']:<3} got {found}")

MISS    Q20 expected s.64  got [74, 103, 115]
MISS    Q21 expected s.75  got [76, 85, 80]
rank 1  Q22 expected s.82  got [82, 99, 56]
MISS    Q23 expected s.90  got [75, 92, 86]
rank 3  Q24 expected s.83  got [89, 94, 83]


In [21]:
# --- Cross-section: answering requires more than one provision ---

item('Q25', 'cross_section',
     'What are my rights if I have children and claim benefits?',
     'plain', 33, 'partial',
     'Sections 33 and 34 both apply. Section 33 prohibits preventing a person '
     'renting on the basis that a child would live with or visit them. Section '
     '34 prohibits the same conduct on the basis that a person is a benefits '
     'claimant. Section 35 renders discriminatory tenancy terms of no effect.')

item('Q26', 'cross_section',
     'Can a landlord assess my affordability but not discriminate against me '
     'for claiming benefits?',
     'plain', 41, 'in_force',
     'Yes. Section 34 prohibits discrimination on the basis of benefits status, '
     'while section 41 provides that nothing in Chapter 3 prohibits taking a '
     'person\'s income into account when considering affordability.')

item('Q27', 'cross_section',
     'What is the complete position on rent paid in advance?',
     'plain', 8, 'partial',
     'Section 8 prohibits rent being due in advance after the tenancy is '
     'entered into, other than initial rent in the permitted pre-tenancy '
     'period. Section 9 makes a payment of rent before the tenancy is entered '
     'into a prohibited payment under the Tenant Fees Act 2019.')

item('Q28', 'cross_section',
     'How do the database provisions affect a landlord seeking possession?',
     'statutory', 90, 'uncommenced',
     'Section 82(3) imposes a duty on a residential landlord to maintain active '
     'landlord and dwelling entries. Section 90 prevents a court making a '
     'possession order while that duty is breached, except on Ground 7A or '
     'Ground 14. Both are enacted but not yet fully commenced.')

item('Q29', 'cross_section',
     'What penalties apply for discrimination against families?',
     'plain', 40, 'partial',
     'Section 40 allows a local housing authority to impose a financial penalty '
     'for breach of section 33 or 34. Section 35 separately renders '
     'discriminatory tenancy terms of no effect, and section 38 renders '
     'corresponding insurance terms of no effect.')

item('Q30', 'cross_section',
     'If my landlord increases the rent, what can I do about it?',
     'plain', 7, 'partial',
     'Section 6 requires a landlord to follow the statutory procedure with two '
     'months notice. Section 7 allows the tenant to apply to the appropriate '
     'tribunal to challenge the proposed new rent.')

In [22]:
for it in testset[24:]:
    hits = retrieve(it['question'], k=3)
    found = [h['section'] for h in hits]
    rank = found.index(it['section']) + 1 if it['section'] in found else None
    mark = f"rank {rank}" if rank else "MISS  "
    print(f"{mark}  {it['id']} expected s.{it['section']:<3} got {found}")

MISS    Q25 expected s.33  got [55, 51, 50]
rank 3  Q26 expected s.41  got [34, 42, 41]
rank 3  Q27 expected s.8   got [103, 56, 8]
MISS    Q28 expected s.90  got [75, 85, 76]
rank 1  Q29 expected s.40  got [40, 44, 50]
rank 1  Q30 expected s.7   got [7, 103, 9]


In [23]:
EXPECTED_MULTI = {
    'Q25': [33, 34, 35],
    'Q26': [34, 41],
    'Q27': [8, 9],
    'Q28': [82, 90],
    'Q29': [40, 35, 38],
    'Q30': [6, 7],
}

for qid, expected in EXPECTED_MULTI.items():
    it = next(x for x in testset if x['id'] == qid)
    found = [h['section'] for h in retrieve(it['question'], k=3)]
    hit = set(expected) & set(found)
    print(f"{qid}: expected {expected} got {found} -> found {sorted(hit)} "
          f"({len(hit)}/{len(expected)})")

Q25: expected [33, 34, 35] got [55, 51, 50] -> found [] (0/3)
Q26: expected [34, 41] got [34, 42, 41] -> found [34, 41] (2/2)
Q27: expected [8, 9] got [103, 56, 8] -> found [8] (1/2)
Q28: expected [82, 90] got [75, 85, 76] -> found [] (0/2)
Q29: expected [40, 35, 38] got [40, 44, 50] -> found [40] (1/3)
Q30: expected [6, 7] got [7, 103, 9] -> found [7] (1/2)


In [24]:
for qid, expected in EXPECTED_MULTI.items():
    it = next(x for x in testset if x['id'] == qid)
    found = [h['section'] for h in retrieve(it['question'], k=5)]
    hit = set(expected) & set(found)
    print(f"{qid}: {len(hit)}/{len(expected)} at k=5  (was k=3)  got {found}")

Q25: 0/3 at k=5  (was k=3)  got [55, 51, 50, 38, 52]
Q26: 2/2 at k=5  (was k=3)  got [34, 42, 41, 35, 55]
Q27: 2/2 at k=5  (was k=3)  got [103, 56, 8, 6, 9]
Q28: 1/2 at k=5  (was k=3)  got [75, 85, 76, 86, 82]
Q29: 1/3 at k=5  (was k=3)  got [40, 44, 50, 57, 66]
Q30: 2/2 at k=5  (was k=3)  got [7, 103, 9, 6, 66]


In [25]:
def oos(qid, question, phrasing, reason):
    testset.append({
        'id': qid, 'theme': 'out_of_scope', 'question': question,
        'phrasing': phrasing, 'section': None, 'commencement': None,
        'gold_answer': f'The Act does not address this. {reason}',
        'answerable': False,
    })

oos('Q31', 'How much notice must my landlord give me to leave?', 'plain',
    'Notice periods are set out in the Housing Act 1988 as amended, not in '
    'the text of this Act.')

oos('Q32', 'What is the current rate of stamp duty on a rental property?',
    'plain', 'Taxation is outside the scope of this Act.')

oos('Q33', 'Can my landlord enter my flat without permission?', 'plain',
    'Quiet enjoyment and access are governed by other legislation.')

oos('Q34', 'What is a good rent for a two-bedroom flat in Lewisham?', 'plain',
    'The Act does not set or indicate rent levels.')

oos('Q35', 'How do I apply for housing benefit?', 'plain',
    'Benefit administration is outside the scope of this Act.')

oos('Q36', 'What is the recipe for carbonara?', 'plain',
    'Entirely unrelated to the Act.')

oos('Q37', 'Does the Act apply to commercial leases?', 'statutory',
    'The Act concerns residential tenancies.')

print(len(testset), "items")

37 items


In [26]:
item('Q38', 'possession',
     'What changes does the Act make to the grounds on which a landlord can '
     'seek possession?',
     'statutory', 3, 'partial',
     'Section 3 amends Schedule 2 to the Housing Act 1988, which sets out the '
     'grounds for possession.')

item('Q39', 'landlord_duties',
     'Does my landlord have to give me anything in writing at the start of a '
     'tenancy?',
     'plain', 12, 'partial',
     'Section 12 imposes a duty on a landlord and contractor to give a '
     'statement of terms and other specified information.')

item('Q40', 'landlord_duties',
     'What penalties can be imposed on a landlord who breaches the duties in '
     'Chapter 1?',
     'statutory', 15, 'partial',
     'Section 15 provides for financial penalties and offences in relation to '
     'landlords and others breaching duties under the Chapter.')

item('Q41', 'guarantor',
     'If a tenant dies, is their guarantor still liable for the rent?',
     'plain', 19, 'partial',
     'Section 19 provides that a guarantor is not liable for rent payable after '
     'the tenant\'s death.')

item('Q42', 'notice_to_quit',
     'How much notice do I have to give my landlord if I want to leave?',
     'plain', 20, 'partial',
     'Section 20 concerns the timing of notices to quit given by tenants under '
     'assured tenancies.')

item('Q43', 'rental_bidding',
     'Can a letting agent invite offers above the advertised rent?',
     'plain', 56, 'partial',
     'Section 56 requires the rent to be stated and prohibits rental bidding.')

item('Q44', 'unlawful_eviction',
     'What are the penalties for unlawfully evicting a tenant?',
     'statutory', 58, 'partial',
     'Section 58 provides for penalties for unlawful eviction or harassment of '
     'an occupier.')

item('Q45', 'decent_homes',
     'Is there a minimum standard my rented home has to meet?',
     'plain', 100, 'partial',
     'Section 100 provides for a decent homes standard.')

item('Q46', 'hazards',
     'My flat has damp and my landlord is ignoring it. What does the Act say?',
     'plain', 60, 'uncommenced',
     'Section 60 concerns the remedying of hazards occurring in dwelling-houses '
     'in England. It is enacted but not yet commenced.')

item('Q47', 'rent_repayment',
     'Can I get my rent back if my landlord commits an offence?',
     'plain', 98, 'partial',
     'Section 98 provides for rent repayment orders in respect of offences '
     'under the Housing Act 1988 and under section 67.')

item('Q48', 'enforcement',
     'Who enforces the provisions of this Act?',
     'plain', 107, 'partial',
     'Section 107 places a general duty of enforcement on local housing '
     'authorities.')

item('Q49', 'homeless',
     'What does the Act change about accommodation for homeless people?',
     'statutory', 25, 'partial',
     'Section 25 concerns the duties of a local authority in relation to '
     'accommodation for homeless people.')

item('Q50', 'agricultural',
     'Does the Act cover agricultural tied accommodation?',
     'statutory', 23, 'partial',
     'Section 23 concerns grounds for possession in relation to assured '
     'agricultural occupancies.')

print(len(testset), "items")

50 items


In [27]:
import json

ts = pd.DataFrame(testset)

with open('../data/processed/rights_testset.json', 'w', encoding='utf-8') as f:
    json.dump(testset, f, indent=2, ensure_ascii=False)

ts.to_csv('../data/processed/rights_testset.csv', index=False)

print("Saved", len(testset), "items")
print()
print("By commencement status:")
print(ts['commencement'].value_counts(dropna=False).to_string())
print()
print("By phrasing:")
print(ts['phrasing'].value_counts().to_string())
print()
print("Answerable:", ts['answerable'].sum(), "| out of scope:", (~ts['answerable'].astype(bool)).sum())

Saved 50 items

By commencement status:
commencement
partial        32
uncommenced     7
NaN             7
in_force        4

By phrasing:
phrasing
plain        32
statutory    18

Answerable: 43 | out of scope: 7


In [28]:
answerable = [it for it in testset if it.get('answerable', True) and it['section']]
hits1 = hits3 = 0
for it in answerable:
    found = [h['section'] for h in retrieve(it['question'], k=3)]
    if it['section'] in found:
        hits3 += 1
        if found[0] == it['section']:
            hits1 += 1

print(f"Recall@1: {hits1}/{len(answerable)} ({hits1/len(answerable)*100:.0f}%)")
print(f"Recall@3: {hits3}/{len(answerable)} ({hits3/len(answerable)*100:.0f}%)")

Recall@1: 19/43 (44%)
Recall@3: 29/43 (67%)
